# N-ATLaS Feasibility + Smoke Test (Google Colab, A100)

Runs a genuine load + multilingual inference test of `NCAIR1/N-ATLaS` on your Colab A100.
No mocked output — this downloads the real (gated) weights and measures load time, latency, and memory.

**Before running:**
1. Runtime → Change runtime type → GPU → A100 (you said this is already selected/running).
2. Accept the license at https://huggingface.co/NCAIR1/N-ATLaS (while logged in).
3. Create a read token at https://huggingface.co/settings/tokens.
4. Add it as a Colab secret named `HF_TOKEN` (key icon in the left sidebar — recommended, survives reruns), or paste it when prompted below.

In [ ]:
!pip install -q -U transformers accelerate safetensors sentencepiece huggingface_hub psutil

In [ ]:
import os

hf_token = None
try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
except Exception:
    pass

if not hf_token:
    import getpass
    hf_token = getpass.getpass('Paste your HF_TOKEN (input hidden): ')

os.environ['HF_TOKEN'] = hf_token
print('HF_TOKEN set:', bool(hf_token))

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,memory.used,driver_version --format=csv

In [ ]:
import platform
import psutil
import torch

print('Python:', platform.python_version())
print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError('No GPU detected. Go to Runtime > Change runtime type and select a GPU (A100) before continuing.')

props = torch.cuda.get_device_properties(0)
print('GPU:', props.name)
print('VRAM (GB):', round(props.total_memory / 1024**3, 2))

vm = psutil.virtual_memory()
print('System RAM total (GB):', round(vm.total / 1024**3, 2))
print('System RAM available (GB):', round(vm.available / 1024**3, 2))

In [ ]:
import time

from huggingface_hub.utils import GatedRepoError, HfHubHTTPError
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME = 'NCAIR1/N-ATLaS'

try:
    t0 = time.perf_counter()
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, token=hf_token)
    tokenizer_load_s = round(time.perf_counter() - t0, 2)
    print('Tokenizer loaded in', tokenizer_load_s, 's')
except (GatedRepoError, HfHubHTTPError) as e:
    raise RuntimeError(
        'Could not access NCAIR1/N-ATLaS. Make sure you have accepted the license at '
        'https://huggingface.co/NCAIR1/N-ATLaS while logged in with the SAME account '
        'that generated your HF_TOKEN, then re-run this cell.'
    ) from e

In [ ]:
t0 = time.perf_counter()
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    token=hf_token,
    torch_dtype=torch.bfloat16,
    device_map='auto',
    low_cpu_mem_usage=True,
)
model_load_s = round(time.perf_counter() - t0, 2)
print('Model loaded in', model_load_s, 's')
print('Peak GPU memory after load (GB):', round(torch.cuda.max_memory_allocated() / 1024**3, 2))

In [ ]:
import json
from datetime import datetime, timezone

def run_prompt(prompt, max_new_tokens=128):
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
    t0 = time.perf_counter()
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
        )
    gen_s = time.perf_counter() - t0
    text = tokenizer.decode(output_ids[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    new_tokens = output_ids.shape[1] - inputs['input_ids'].shape[1]
    return {
        'prompt': prompt,
        'generated_text': text,
        'new_tokens': int(new_tokens),
        'generation_seconds': round(gen_s, 2),
        'tokens_per_second': round(new_tokens / gen_s, 2) if gen_s > 0 else None,
    }

prompts = [
    (
        'english_health',
        'You are N-ATLaS, a multilingual Nigerian AI assistant. Respond briefly in English.\n\n'
        'Question: What are three ways a rural health worker in Nigeria could use an AI '
        'assistant during a facility stock report?\nAnswer:'
    ),
    (
        'multilingual_check',
        'You are N-ATLaS, a multilingual Nigerian AI assistant. '
        'Please answer the following question in Igbo, then repeat the answer in Yoruba, '
        'then repeat it in Hausa, clearly labelling each language.\n\n'
        'Question: How are you today?\nAnswer:'
    ),
]

results = {
    'timestamp_utc': datetime.now(timezone.utc).isoformat(),
    'model_name': MODEL_NAME,
    'device': 'cuda',
    'gpu_name': props.name,
    'gpu_vram_gb': round(props.total_memory / 1024**3, 2),
    'tokenizer_load_seconds': tokenizer_load_s,
    'model_load_seconds': model_load_s,
    'peak_gpu_memory_load_gb': round(torch.cuda.max_memory_allocated() / 1024**3, 2),
    'runs': {},
}

for key, p in prompts:
    print(f'--- Running: {key} ---')
    r = run_prompt(p)
    print(r['generated_text'])
    print()
    results['runs'][key] = r

results['peak_gpu_memory_overall_gb'] = round(torch.cuda.max_memory_allocated() / 1024**3, 2)
print(json.dumps(results, indent=2))

In [ ]:
with open('natlas_colab_result.json', 'w') as f:
    json.dump(results, f, indent=2)

from google.colab import files
files.download('natlas_colab_result.json')

Download `natlas_colab_result.json` and share it back — it gets folded into `docs/FEASIBILITY.md` as the real, measured result (including the multilingual check, which doubles as N-ATLaS integration evidence for the submission).